In [1]:
import json
import os
import sys
from pathlib import Path
from datetime import datetime
import hashlib

import wandb
from dotenv import load_dotenv
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

sys.path.append(os.path.abspath("../.."))

from src.utils.get_trainer import get_trainer
from src.utils.loggers import WandbLogger
from src.utils.telegram import send_message

# Create OOF and Test pred

In [2]:
# ===== User config =====
load_dotenv(dotenv_path="../../.env")

model_name = "xgb"
data_id = "033"

n_trial = 17
n_custom = None

n_fold = 5
seed = 42

In [4]:
# === get params, n_fold, seed, and batch_rows ===
study_name = f"{model_name}-{data_id}"
feature_dir = Path(f"../../artifacts/features/{data_id}")

if n_trial is not None and n_custom is not None:
    raise ValueError("Set either n_trial or custom_params_json (not both).")
if n_trial is not None:
    params_id = f"trl{n_trial}"
    params_path = f"../../artifacts/params/{study_name}/{params_id}.json"
elif n_custom is not None:
    params_id = f"cus{n_custom}"
    params_path = f"../../artifacts/params/custom/{params_id}.json"
else:
    raise ValueError("Set n_trial or custom_params_json.")

with open(feature_dir / "meta.json", "r") as f:
    meta = json.load(f)

train_paths = meta["train_paths"]
test_paths = meta["test_paths"]
level = meta["level"]

with open(params_path, "r") as f:
    manifest = json.load(f)

params = manifest["params"]
opts = manifest["opts"]

print("Manifest path: ", params_path)
print("Params:\n", params)
print("opts: ", opts)

Manifest path:  ../../artifacts/params/xgb-033/trl17.json
Params:
 {'learning_rate': 0.02, 'max_depth': 8, 'min_child_weight': 95.07143064099162, 'colsample_bytree': 0.592797576724562, 'subsample': 0.7394633936788146, 'reg_alpha': 0.0007482139197236472, 'reg_lambda': 0.000602521573620386}
opts:  {}


In [5]:
# === WANDB ===
wandb_project = os.environ.get("COMPETITION_NAME")
wandb.login(key=os.environ.get("WANDB_API_KEY"))

run = wandb.init(
    project=wandb_project,
    group=study_name,
    name=params_id,
    job_type="cv_training",
    tags=[model_name, level],
    config={
        "data_id": data_id,
        "n_fold": n_fold,
        "seed": seed,
        **params,
        **opts
    }
)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /home/hanse/.netrc
wandb: Currently logged in as: kaitookano (kaitookano-waseda-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
# === Training ===
trainer_class = get_trainer(model_name)
trainer = trainer_class(
    data_id,
    train_paths,
    test_paths,
    features=None,
    target="target",
    fold_col=None,
    weight_col=None,
    cat_cols=None,
    params=params,
    n_fold=n_fold,
    seed=seed,
    gpu=True,
    opts=opts
)

result = trainer.fit(
    loggers=[WandbLogger(run=run)]
)

Fold Col: 5fold-s42
===== Fold 1 / 5 =====
Free CPU Mem: 18.13 GB
Free GPU Mem: 6.75 GB
[0]	train-auc:0.96159	eval-auc:0.96155
[100]	train-auc:0.97308	eval-auc:0.97231
[200]	train-auc:0.97480	eval-auc:0.97366
[300]	train-auc:0.97599	eval-auc:0.97447
[400]	train-auc:0.97694	eval-auc:0.97503
[500]	train-auc:0.97768	eval-auc:0.97533
[600]	train-auc:0.97830	eval-auc:0.97555
[700]	train-auc:0.97885	eval-auc:0.97571
[800]	train-auc:0.97935	eval-auc:0.97583
[900]	train-auc:0.97978	eval-auc:0.97593
[1000]	train-auc:0.98021	eval-auc:0.97601
[1100]	train-auc:0.98057	eval-auc:0.97607
[1200]	train-auc:0.98092	eval-auc:0.97611
[1300]	train-auc:0.98129	eval-auc:0.97615
[1400]	train-auc:0.98163	eval-auc:0.97621
[1500]	train-auc:0.98197	eval-auc:0.97624
[1600]	train-auc:0.98229	eval-auc:0.97627
[1700]	train-auc:0.98262	eval-auc:0.97630
[1800]	train-auc:0.98292	eval-auc:0.97632
[1900]	train-auc:0.98323	eval-auc:0.97634
[2000]	train-auc:0.98352	eval-auc:0.97636
[2100]	train-auc:0.98383	eval-auc:0.97637


In [ ]:
# ===== Save Manifest, OOF, and Test pred=====
runs_root = Path("../../runs")
run_id = f"{model_name}-{data_id}-{params_id}-{n_fold}fold-s{seed}"
run_dir = runs_root / run_id
run_dir.mkdir(parents=True, exist_ok=True)

s = json.dumps(params, sort_keys=True, separators=(",", ":"), ensure_ascii=False)
phash = hashlib.sha256(s.encode()).hexdigest()[:10]

manifest = {
    "run_id": run_id,
    "wandb_id": run.id,
    "wandb_url": run.url,
    "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "model_name": model_name,
    "data_id": data_id,
    "n_fold": n_fold,
    "seed": seed,
    "feature_dir": str(feature_dir.resolve()),
    "phash": phash,
    "param_source": params_path,
    "params": params,
    "run_dir": str(run_dir.resolve()),
    "cv_score": result.oof_score,
    "submission": {
        "competition": None,
        "ref": None,
        "file": None,
        "public_score": None,
        "private_score": None,
        "submitted_at": None
    }
}

with open(run_dir / "manifest.json", "w") as f:
    json.dump(manifest, f, indent=4)

print("manifest saved", run_dir / "manifest.json")

# oof and test pred
np.save(run_dir / "oof.npy", result.oof)
np.save(run_dir / "test.npy", result.test_pred)

print(f"oof and test preds saved successfully in:\n{run_dir}")
send_message(
    f"✅ Finished CV for {run_id}!"
    f"\nScore: {round(result.oof_score, 5)}"
)

# Check Feature Importance (Only XGB)

In [ ]:
# Plot feature importance and get top columns
df = result.fi_mean.to_pandas()
df_top100 = df[:100]

fig, ax = plt.subplots(figsize=(24, 32))
sns.barplot(
    data=df_top100,
    y="Feature",
    x="mean_ratio",
    orient="h",
    palette="flare",
    hue="Feature",
    ax=ax
)
for container in ax.containers:
    labels = ax.bar_label(container)
    for label in labels:
        label.set_fontsize(20)
plt.title("Feature Importance Top 100", fontsize=32)
plt.xlabel("Importance", fontsize=28)
plt.ylabel("Feature", fontsize=28)
ax.tick_params(axis="x", labelsize=20)
ax.tick_params(axis="y", labelsize=20)
plt.tight_layout()

fig.savefig(run_dir / "feature_importance.png", dpi=200, bbox_inches="tight")
fig.savefig(run_dir / "feature_importance.svg", bbox_inches="tight")

plt.show()

print(f"FI graph saved successfully to {run_dir}")

In [ ]:
thresholds = [0.90, 0.95, 0.99]       # しきい値リスト

total = df["mean_ratio"].sum()
print("All cols ", len(df))

out = {}
for th in thresholds:
    limit = th * total                # このしきい値に対応する累積比率

    # 累積と K を計算（「閾値を超える要素も1つ含める」ロジック）
    cum = df["mean_ratio"].cumsum()
    k = int((cum <= limit).sum())
    if k < len(df):
        k += 1

    drop_cols = df["Feature"].iloc[k:].tolist()
    print(f"\nThreshold {th:.2f} → Drop {len(drop_cols)} cols")
    out[str(k)] = drop_cols

path = run_dir / "drop_cols.json"
with open(path, "w", encoding="utf-8") as f:
    json.dump(out, f, ensure_ascii=False, indent=4)
print(f"saved: {path}")